# Carregamento do CSV de exemplo + Treinamento do modelo de risco de alagamento

**Projeto Integrador Univesp – PIV420 – Sorocaba**

Este notebook:
1. Carrega o arquivo `dados_exemplo_alagamento_sorocaba.csv`
2. Explora as variáveis e unidades
3. Separa treino/teste
4. Treina um modelo **Random Forest**
5. Avalia o desempenho
6. Salva o modelo treinado
7. Faz um exemplo de previsão

### Unidades
| Coluna | Unidade |
|--------|---------|
| chuva_1h, chuva_3h, chuva_6h, chuva_24h | mm (milímetros) |
| nivel_rio | m (metros) |
| risco | 0 = Baixo, 1 = Médio, 2 = Alto |


## 1. Importação das bibliotecas

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib
import matplotlib.pyplot as plt

print("Bibliotecas carregadas.")


## 2. Carregamento dos dados

In [ ]:
# Ajuste o caminho se necessário
caminho = Path("dados_exemplo_alagamento_sorocaba.csv")
# Alternativa no ambiente do projeto:
if not caminho.exists():
    caminho = Path("/home/workdir/artifacts/dados_exemplo_alagamento_sorocaba.csv")

df = pd.read_csv(caminho)
df["data_hora"] = pd.to_datetime(df["data_hora"])

print("Dimensões:", df.shape)
print("\nUnidades:")
print("  chuva_*   → mm (milímetros)")
print("  nivel_rio → m (metros)")
print("  risco     → 0=Baixo, 1=Médio, 2=Alto")
print("\nPrimeiras linhas:")
df.head()


## 3. Estatísticas descritivas e distribuição do risco

In [ ]:
print("Estatísticas das variáveis numéricas:")
display(df[["chuva_1h", "chuva_3h", "chuva_6h", "chuva_24h", "nivel_rio"]].describe().round(2))

print("\nDistribuição do risco:")
print(df["risco"].value_counts().sort_index().rename({0: "Baixo (0)", 1: "Médio (1)", 2: "Alto (2)"}))


## 4. Separação treino / teste

In [ ]:
features = ["chuva_1h", "chuva_3h", "chuva_6h", "chuva_24h", "nivel_rio"]
X = df[features]
y = df["risco"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"Treino: {len(X_train)} registros")
print(f"Teste:  {len(X_test)} registros")
print(f"Features: {features}")


## 5. Treinamento do modelo (Random Forest)

In [ ]:
modelo = RandomForestClassifier(
    n_estimators=150,
    class_weight="balanced",  # equilibra classes desbalanceadas
    random_state=42,
)
modelo.fit(X_train, y_train)
print("Modelo treinado com sucesso.")


## 6. Avaliação

In [ ]:
y_pred = modelo.predict(X_test)

print("Acurácia:", round(accuracy_score(y_test, y_pred), 4))
print("\nRelatório de classificação:")
print(classification_report(y_test, y_pred, target_names=["Baixo", "Médio", "Alto"]))
print("Matriz de confusão:")
print(confusion_matrix(y_test, y_pred))


## 7. Importância das variáveis

In [ ]:
importancias = pd.Series(
    modelo.feature_importances_, index=features
).sort_values(ascending=False)

print("Importância das variáveis:")
print(importancias.round(3))

# Gráfico
fig, ax = plt.subplots(figsize=(7, 3.5))
importancias.sort_values().plot(kind="barh", ax=ax, color="#2980b9")
ax.set_title("Importância das variáveis (Random Forest)")
ax.set_xlabel("Importância")
plt.tight_layout()
plt.show()


## 8. Salvar o modelo treinado

In [ ]:
caminho_modelo = Path("modelo_risco_alagamento.pkl")
joblib.dump(modelo, caminho_modelo)
print(f"Modelo salvo em: {caminho_modelo.resolve()}")


## 9. Exemplo de previsão

Informe valores hipotéticos de chuva (mm) e nível do rio (m) para obter a classificação de risco.


In [ ]:
# chuva_1h, chuva_3h, chuva_6h, chuva_24h, nivel_rio
exemplo = np.array([[5.0, 15.0, 30.0, 65.0, 3.2]])
pred = modelo.predict(exemplo)[0]
proba = modelo.predict_proba(exemplo)[0]
rotulo = {0: "Baixo", 1: "Médio", 2: "Alto"}[pred]

print(f"Entrada: chuva_1h=5.0, chuva_3h=15.0, chuva_6h=30.0, chuva_24h=65.0, nivel_rio=3.2")
print(f"Risco previsto: {rotulo} (código {pred})")
print(f"Probabilidades → Baixo: {proba[0]:.2%} | Médio: {proba[1]:.2%} | Alto: {proba[2]:.2%}")


## Observação

Este CSV é **de exemplo** para demonstrar o fluxo de machine learning no protótipo acadêmico.

Na análise com dados reais (SIBH + eventos documentados), concluiu-se que ainda faltam série contínua de nível do rio e mais eventos rotulados para um modelo supervisionado robusto em operação. O dashboard do projeto também utiliza **regras de limiar** como abordagem principal e transparente.
